# A/B Testing – Warm-up (15 minutes) ⏱️

A short exercise to try the tool while the lecture is still fresh. Three tasks, one tool: `statsmodels`.

**Scenario:** ShopNow wants more newsletter sign-ups. Today **2.0 %** of visitors sign up.
Marketing proposes a new sign-up banner (variant B) and hopes for **+0.4 percentage points** (2.0 % → 2.4 %).
The site has about **3,000 visitors per day**.

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest, confint_proportions_2indep

BASELINE = 0.02
MDE = 0.004
VISITORS_PER_DAY = 3000

## Task 1: How many users do we need?

In [ ]:
effect = proportion_effectsize(BASELINE + MDE, BASELINE)
n_per_group = NormalIndPower().solve_power(effect_size=effect, alpha=0.05, power=0.8, alternative="two-sided")
days = 2 * n_per_group / VISITORS_PER_DAY
print(f"{n_per_group:,.0f} users per group -> {days:.0f} days")

## Task 2: Marketing wants an answer in one week

They say: "Just run it for 7 days." What is the smallest lift (in percentage points) you could detect with 80 % power in 7 days?
What do you tell them?

In [ ]:
import numpy as np
n_week = 7 * VISITORS_PER_DAY / 2
for mde in np.arange(0.001, 0.03, 0.001):
    es = proportion_effectsize(BASELINE + mde, BASELINE)
    power = NormalIndPower().power(effect_size=es, nobs1=n_week, alpha=0.05, ratio=1.0)
    if power >= 0.8:
        print(f"in 7 days ({n_week:,.0f} users per group) we can detect at least +{mde*100:.1f} pp (power {power:.0%})")
        break
# What to tell them: a one-week test only finds lifts of about +0.6 pp or more (that is +30 % relative!).
# If the banner "only" delivers the hoped-for +0.4 pp, a 7-day test has well under 80 % power and will often miss it.
# Either run the full two weeks or agree upfront that only a big win counts.

## Task 3: The test ran. Decide.

After the planned duration you get the numbers from the data warehouse:

| variant | visitors | sign-ups |
|---|---|---|
| A (old banner) | 48,200 | 965 |
| B (new banner) | 48,050 | 1,160 |

In [ ]:
visitors = [48_200, 48_050]   # A, B
signups = [965, 1_160]        # A, B

rate_a, rate_b = signups[0] / visitors[0], signups[1] / visitors[1]
z, p = proportions_ztest(count=signups, nobs=visitors)
low, high = confint_proportions_2indep(signups[1], visitors[1], signups[0], visitors[0])

print(f"A: {rate_a:.2%}, B: {rate_b:.2%}")
print(f"lift B - A: {(rate_b - rate_a)*100:+.2f} pp, 95 % CI [{low*100:+.2f}, {high*100:+.2f}] pp, p = {p:.2g}")
# Decision: the lift is about +0.4 pp, the interval excludes 0 and p is far below 0.05.
# The effect is real and as large as planned -> ship the new banner, provided the guardrails
# (unsubscribes, page load time) did not get worse.

**Bonus (if you are fast):** what happens to the p-value if both groups had only 4,800 visitors with the same rates? Try it.